# Indic Speech AI Research Assignment — Submission Notebook

This is the cleared Colab runbook for the Task 2 implementation and evaluation. The code and results are also in the project source folders and aggregate `results/` files. Long inference cells are deliberately not auto-executed; do not use **Run all** without selecting a suitable GPU runtime and reading the execution notes.

## Recorded results at a glance

- **FLEURS test:** 50 Hindi + 50 Kannada source utterances, each clean/noisy; greedy, fixed beam-5, and adaptive beam-5. Macro test WER/CER: greedy 86.76%/73.64%; adaptive threshold -0.3 84.83%/72.31%; fixed beam-5 83.02%/70.62%.
- **Research extension:** threshold was selected on validation only. The chosen -0.3 threshold improves macro WER over greedy but does not beat fixed beam-5. The selection process and per-condition tables are in `FINAL_RESEARCH_REPORT.md` and `results/`.
- **End-to-end QA:** 100 Hindi synthetic-TTS examples, Indic-Transcribe Core + Qwen2.5-1.5B-Instruct. CPU measurements: transcript WER 17.73%, CER 7.98%, mean pipeline latency 202.47 s, mean pipeline RTF 14.502.
- **Answer-quality labels:** 12/100 correct/adequate, 24/100 hallucination flags, 44/100 visible-truncation flags are provisional AI-assisted estimates, not independent human labels. The original `human_*` review fields remain blank.
- **Reproducibility:** 60/60 paired transcripts matched in a 10-utterance spot check.

## Run order

1. Run the setup cell to clone `https://github.com/sajid-mohd/indic-speech-ai-task2-.git` into `/content/speech_ai_assignment`. If that directory already contains the expected project files, the cell reuses it. ZIP upload is not required.
2. Use a GPU runtime for the FLEURS/Whisper and Indic-Transcribe tests. Gated model access and a Hugging Face login are required for Indic-Transcribe.
3. Run smoke tests before full ASR experiments. Run one deterministic FLEURS validation pass, analyze thresholds offline, then run the held-out test only after freezing the threshold.
4. The final 100-example QA measurements in this report were made on CPU (float32) because GPU access was unavailable then; the run took several hours. Re-run that cell only when a new inference run is genuinely needed.
5. Response-quality rates in this release are AI-assisted estimates, not human-verified. A 20-example spot-check found at least one clear question/reference-answer mismatch and several multi-task prompts with potentially incomplete references. The spot-check did not establish a general CSV row-order shift. Treat the rates as exploratory until ambiguous examples are handled using a documented rule and response labels are independently reviewed.

**Security:** never commit Hugging Face or GitHub tokens. Raw per-example QA predictions from the gated dataset are retained privately in Drive and intentionally omitted from the shareable project ZIP; only aggregate results are included.


## Verified result snapshot (no inference runs in this cell)

This snapshot is for review convenience; full methodology and interpretation are in `FINAL_RESEARCH_REPORT.md`.

| Dataset/metric | Observed result |
|---|---:|
| FLEURS test macro WER — greedy / adaptive / fixed beam-5 | 86.76% / 84.83% / **83.02%** |
| FLEURS test macro CER — greedy / adaptive / fixed beam-5 | 73.64% / 72.31% / **70.62%** |
| Adaptive threshold selected on validation | -0.3 (fallback constraint ≤50%) |
| QA questions evaluated | 100 Hindi synthetic-TTS examples |
| QA transcript WER / CER | 17.73% / 7.98% |
| QA mean pipeline latency / RTF | 202.47 s / 14.502 (CPU) |
| QA outputs at token limit | 44/100 (44%) |
| QA response-quality estimates | 12 correct/adequate; 24 hallucination flags; 44 visibly incomplete — AI-assisted, not independently human-verified |


In [ ]:
# PHASE 1 — Clone the project from GitHub or reuse the existing project folder.

from pathlib import Path
import subprocess

REPO_URL = "https://github.com/sajid-mohd/indic-speech-ai-task2-.git"
PROJECT_DIR = Path("/content/speech_ai_assignment")
EXPECTED_SCRIPT = PROJECT_DIR / "scripts" / "evaluate_indic_transcribe.py"
EXPECTED_REQUIREMENTS = PROJECT_DIR / "requirements-colab.txt"

if EXPECTED_SCRIPT.is_file() and EXPECTED_REQUIREMENTS.is_file():
    print("Existing project found:", PROJECT_DIR)
    if (PROJECT_DIR / ".git").exists():
        print("Git checkout detected. To fetch newer changes, run `git pull` in this directory before continuing.")
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(
        f"{PROJECT_DIR} exists but does not contain the expected project files. "
        "Inspect the directory before removing or replacing anything. "
        "If this is an existing Git checkout, update it manually and rerun this cell."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
    print("Repository cloned successfully.")

if not EXPECTED_SCRIPT.is_file():
    raise FileNotFoundError(f"Expected project script was not found: {EXPECTED_SCRIPT}")
if not EXPECTED_REQUIREMENTS.is_file():
    raise FileNotFoundError(f"Colab requirements file was not found: {EXPECTED_REQUIREMENTS}")

print("Project directory:", PROJECT_DIR)
print("Colab package requirements:", EXPECTED_REQUIREMENTS)
print("Setup complete.")


In [ ]:
# PHASE 1 — Install the compatible packages ONCE.
# Keeps Colab-compatible versions of Hub, Transformers, Datasets, and fsspec.
# The requirements deliberately do not replace the CUDA-enabled torch/torchaudio builds.
%pip install -q -r /content/speech_ai_assignment/requirements-colab.txt
print("Dependency installation finished. Now restart the Colab runtime before proceeding.")

## CELL 3 — Mandatory restart checkpoint

After Cell 2 completes, click **Runtime → Restart session**. This prevents the notebook from mixing old imports with newly installed package files. After the restart, continue at Cell 4. Do not rerun Cells 1–2 unless you intentionally rebuild the environment.

In [ ]:
# PHASE 2 — Verify the runtime after restart.
import os, sys, subprocess, importlib.util, json, time, gc
from pathlib import Path
import torch

PROJECT_DIR = Path("/content/speech_ai_assignment")
if not PROJECT_DIR.exists():
    raise FileNotFoundError("Project directory is missing. Rerun the GitHub setup cell first, then install dependencies and restart the runtime.")
os.chdir(PROJECT_DIR)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → T4 GPU, then restart.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

# PACKAGE_VERSION_CHECK: confirm the modern Colab stack was not downgraded.
from importlib.metadata import version, PackageNotFoundError
for package in ["huggingface-hub", "transformers", "datasets", "fsspec", "faster-whisper", "av"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


In [ ]:
# PHASE 2 — Mount Drive, restore previous results, and configure durable checkpoints.
from google.colab import drive
import shutil, time
drive.mount('/content/drive')
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/speech_ai_assignment_backup')
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR = DRIVE_BACKUP_DIR / 'results'
LOCAL_RESULTS_DIR = PROJECT_DIR / 'results'
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# Restore previous result files only if this runtime's results folder is empty.
local_result_files = [p for p in LOCAL_RESULTS_DIR.rglob('*') if p.is_file() and p.name != '.gitkeep']
drive_result_files = list(DRIVE_RESULTS_DIR.rglob('*')) if DRIVE_RESULTS_DIR.exists() else []
if not local_result_files and DRIVE_RESULTS_DIR.exists():
    shutil.copytree(DRIVE_RESULTS_DIR, LOCAL_RESULTS_DIR, dirs_exist_ok=True)
    print('Restored prior results from Drive.')
else:
    print(f'Keeping local results ({len(local_result_files)} files); Drive restore skipped.')
def sync_results_to_drive():
    """Copy result/checkpoint files to Drive; source code remains in the ZIP."""
    if LOCAL_RESULTS_DIR.exists():
        shutil.copytree(LOCAL_RESULTS_DIR, DRIVE_RESULTS_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.tmp.csv'))
    print('Results/checkpoints synced to Drive:', DRIVE_RESULTS_DIR)
# Save a source snapshot for reference; never store HF tokens in the project.
shutil.copy2(PROJECT_DIR / 'README.md', DRIVE_BACKUP_DIR / 'README.md')
print('Drive backup ready:', DRIVE_BACKUP_DIR)
print('Call sync_results_to_drive() after each completed experiment; the runner below also syncs periodically.')


In [ ]:
# PHASE 2 — Configure CUDA 12 library paths before launching evaluation subprocesses.
# Do not import ctranslate2/faster_whisper in this notebook kernel before this setup.
import importlib.util, os
from pathlib import Path

def find_nvidia_libdir(package_name, required_library):
    spec = importlib.util.find_spec(package_name)
    if spec is None or not spec.submodule_search_locations:
        raise ImportError(f"Package {package_name} is not installed. Rerun Cell 2 and restart.")
    for location in spec.submodule_search_locations:
        candidate = Path(location) / "lib"
        if (candidate / required_library).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {required_library} under {list(spec.submodule_search_locations)}"
    )

cublas_dir = find_nvidia_libdir("nvidia.cublas", "libcublas.so.12")
cudnn_dir = find_nvidia_libdir("nvidia.cudnn", "libcudnn.so.9")
old_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
paths = [str(cublas_dir), str(cudnn_dir)] + ([old_ld_path] if old_ld_path else [])
os.environ["LD_LIBRARY_PATH"] = ":".join(paths)
print("cuBLAS:", cublas_dir / "libcublas.so.12")
print("cuDNN:", cudnn_dir / "libcudnn.so.9")
print("LD_LIBRARY_PATH configured for fresh Python subprocesses.")

In [ ]:
# PHASE 2 — Test PyAV + faster-whisper + CTranslate2 GPU in a FRESH process.
# A child process is intentional: LD_LIBRARY_PATH must be set before Python starts.
probe = r"""
import av, torch, ctranslate2, faster_whisper
print("PyAV version:", av.__version__)
print("faster-whisper import: OK")
print("CTranslate2 version:", ctranslate2.__version__)
print("PyTorch CUDA available:", torch.cuda.is_available())
if int(av.__version__.split(".")[0]) >= 18:
    raise RuntimeError("PyAV 18+ is incompatible with faster-whisper 1.2.1; pin av<18 and restart.")
count = ctranslate2.get_cuda_device_count()
print("CTranslate2 CUDA devices:", count)
if count < 1:
    raise RuntimeError("CTranslate2 cannot see a CUDA device. Check CUDA library paths and GPU runtime.")
print("Supported CUDA compute types:", ctranslate2.get_supported_compute_types("cuda"))
"""
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_DIR / "src")
check = subprocess.run([sys.executable, "-c", probe], cwd=PROJECT_DIR, env=env, text=True, capture_output=True)
print(check.stdout)
if check.stderr:
    print(check.stderr)
if check.returncode != 0:
    raise RuntimeError("CUDA/dependency probe failed; do not run the expensive evaluation yet.")

In [ ]:
# PHASE 2 — Sign in to Hugging Face.
from huggingface_hub import notebook_login, HfApi
notebook_login()
who = HfApi().whoami()
print("Authenticated Hugging Face account:", who.get("name", "authenticated"))

In [ ]:
# PHASE 2 — Confirm approved access to the gated Indic-Transcribe Core model.
from huggingface_hub import HfApi

MODEL_ID = "bodhan-ai/indic-transcribe-core"
try:
    model_info = HfApi().model_info(MODEL_ID)
    print("Model access confirmed:", model_info.id)
    print("Exact model revision:", model_info.sha)
except Exception as exc:
    print(type(exc).__name__ + ":", str(exc)[:1000])
    raise RuntimeError(
        "Model metadata access failed. Confirm that access was approved for this Hugging Face account."
    ) from exc

In [ ]:
# Helpers: stream long runs and sync result checkpoints to Drive every 60 seconds.
def run_project(args):
    import subprocess, time, os, sys
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_DIR / 'src')
    print('\n$', sys.executable, *args, flush=True)
    proc = subprocess.Popen([sys.executable, '-u', *args], cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    last_sync = time.monotonic()
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            if time.monotonic() - last_sync >= 60:
                try:
                    sync_results_to_drive()
                    last_sync = time.monotonic()
                except Exception as exc:
                    print('WARNING: Drive checkpoint sync failed:', repr(exc), flush=True)
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        try: sync_results_to_drive()
        except Exception as exc: print('Final sync warning:', repr(exc))
        raise
    if code != 0:
        try: sync_results_to_drive()
        except Exception as exc: print('Failure checkpoint sync warning:', repr(exc))
        raise subprocess.CalledProcessError(code, args)
    sync_results_to_drive()
    print('Run completed and results synced to Drive.')

run_project(['-m', 'pytest', '-q'])


## ASR smoke test

Start with only **3 Hindi + 3 Kannada utterances**. Do not run the 100-utterance test until this succeeds and produces the summary CSV. The script performs real inference using Indic-Transcribe Core and Whisper on the same clean/noisy audio examples.

In [ ]:
# PHASE 3 — Smoke test (3 utterances per language).
run_project([
    "scripts/evaluate_indic_transcribe.py",
    "--split", "validation",
    "--n-per-language", "3",
    "--seed", "42",
    "--device", "cuda",
    "--compute-type", "float16",
    "--output-dir", "results/indic_smoke",
])

In [ ]:
# PHASE 3 — Inspect smoke-test results before starting the larger experiment.
import pandas as pd
summary_path = PROJECT_DIR / "results/indic_smoke/indic_vs_whisper_validation_summary.csv"
pred_path = PROJECT_DIR / "results/indic_smoke/indic_vs_whisper_validation_predictions.csv"
if not summary_path.exists() or not pred_path.exists():
    raise FileNotFoundError("Smoke-test result files were not produced. Do not continue yet.")
summary_df = pd.read_csv(summary_path)
pred_df = pd.read_csv(pred_path)
display(summary_df)
print("Prediction rows:", len(pred_df))
print("Models:", sorted(pred_df["model"].dropna().unique().tolist()))
print("Conditions:", sorted(pred_df["condition"].dropna().unique().tolist()))

## Full ASR comparison

Run this only after the smoke test works. It evaluates 50 distinct utterances per language on the **test split**, under clean and deterministic synthetic 10 dB white-noise conditions. The result runner syncs outputs to Drive at completion. Do not use test results to tune decoding thresholds.


## Research extension: confidence-triggered adaptive decoding

The FLEURS runner checkpoints each completed utterance to a partial CSV; the helper syncs files/checkpoints to Google Drive. Whisper decoding now sets `temperature=0.0` explicitly so the baseline and beam-search passes do not fall back to non-zero-temperature sampling. That default temperature schedule was the cause of baseline prediction drift across the previous threshold runs.

**Run one validation inference pass only.** It stores the greedy and fixed beam-5 transcript, confidence proxy, and per-pass latency for every clip/condition. Then the offline analyzer derives thresholds −0.4, −0.6, and −0.8 from these same saved predictions. This makes the threshold comparison paired and requires no further model inference to compare thresholds. If Colab is reclaimed, restore the project/results from Drive and rerun the same command/output directory to resume; do not change split, seed, sample count, or model.


In [ ]:
# PHASE 4 — One deterministic validation inference pass (100 unique clips).
# Threshold argument only populates the runner's native adaptive row; the offline
# analyzer below derives ALL three thresholds from the same greedy + beam outputs.
run_project([
    "scripts/evaluate_fleurs.py",
    "--split", "validation",
    "--n-per-language", "50",
    "--seed", "42",
    "--adaptive-threshold", "-0.6",
    "--device", "cuda",
    "--compute-type", "float16",
    "--output-dir", "results/whisper_validation_deterministic",
])


In [ ]:
# PHASE 4 — Derive and compare thresholds offline; this cell performs NO model inference.
run_project([
    "scripts/analyze_adaptive_thresholds.py",
    "--input-csv", "results/whisper_validation_deterministic/fleurs_validation_predictions.csv",
    "--thresholds", "-1.0", "-0.8", "-0.7", "-0.6", "-0.5", "-0.4", "-0.3", "-0.2", "-0.1", "0.0",
    "--output-dir", "results/whisper_validation_threshold_analysis",
])


In [ ]:
# PHASE 4 — Inspect validation-only threshold comparison before freezing a threshold.
from pathlib import Path
import pandas as pd
macro_path = PROJECT_DIR / "results/whisper_validation_threshold_analysis/validation_threshold_macro_summary.csv"
strata_path = PROJECT_DIR / "results/whisper_validation_threshold_analysis/validation_threshold_summary.csv"
if not macro_path.exists() or not strata_path.exists():
    raise FileNotFoundError("Run the deterministic validation pass and offline threshold analysis first.")
print("Macro summary: equal-weight average across Hindi/Kannada × clean/noisy strata.")
print("Use macro WER as the primary criterion; use CER, latency, RTF and fallback rate to describe the trade-off.")
display(pd.read_csv(macro_path))
print("Per-stratum results:")
display(pd.read_csv(strata_path))
print("\nDo not choose from held-out test results. Record your selected threshold and rationale in the report.")


In [ ]:
# Freeze the threshold selected on validation; do not select from the test set.
# Selection rule: among thresholds with fallback rate <= 50%, choose lowest validation macro WER.
FINAL_THRESHOLD = -0.3
ALLOWED_THRESHOLDS = (-1.0, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0.0)
if FINAL_THRESHOLD not in ALLOWED_THRESHOLDS:
    raise ValueError("FINAL_THRESHOLD must be one of the thresholds in the validation-only sweep.")
print(f"Frozen validation-selected threshold: {FINAL_THRESHOLD}")
print("Rationale: lowest validation macro WER among tested thresholds with fallback <= 50%.")


## Final locked FLEURS test suite

The validation threshold is now frozen. Run the held-out FLEURS test subset once. The test runner also uses explicit `temperature=0.0`; do not change model settings or reselect the threshold after inspecting test output.


In [ ]:
# PHASE 5 — Final locked Whisper decoding ablation on held-out FLEURS test data.
# This reproduces the run whose aggregate results are included in results/asr_test_summary.csv.
run_project([
    "scripts/evaluate_fleurs.py",
    "--split", "test",
    "--model-size", "small",
    "--n-per-language", "50",
    "--languages", "hi_in", "kn_in",
    "--seed", "42",
    "--noise-snr-db", "10",
    "--adaptive-threshold", str(FINAL_THRESHOLD),
    "--device", "cuda",
    "--compute-type", "float16",
    "--output-dir", "results/final_test_threshold_m03",
])


## End-to-end speech → text response benchmark

The dataset `Pastaaaaa2003/Hindi-speech-instruct` requires Hugging Face authentication and approved access. The recorded run used the validation split at dataset revision `3dc2ceb4f564d25f673d446fe8533e1ed6472a50`; the evaluator checks available splits at runtime. The question audio is generated with MMS Hindi TTS, so it is controlled synthetic-speech evidence rather than natural-speaker evidence.

The completed 100-example run used Indic-Transcribe Core + Qwen2.5-1.5B-Instruct on CPU (`float32`) because GPU access was unavailable at the time. It recorded question transcript corpus WER 17.73%, CER 7.98%, mean pipeline latency 202.47 s, mean pipeline RTF 14.502, and 44/100 responses reaching the 192-token generation cap. Aggregate results are in `results/qa_run_summary.json`.


## Q&A quality pilot

The five-example pilot completed successfully on the GPU. Two of five responses reached the 192-token generation cap; its small sample is a diagnostic only. Review relevance, language, and token-cap warnings before attempting another full run. The final 100-example run is already complete and saved; do not rerun it unless reproducing the experiment.


In [ ]:
# PHASE 6 — Five-example Q&A pilot; separate output directory.
run_project([
    "scripts/evaluate_speech_qa.py",
    "--n-samples", "5",
    "--seed", "42",
    "--split", "validation",
    "--asr-backend", "indic-transcribe",
    "--llm-model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--max-new-tokens", "192",
    "--device", "cuda",
    "--compute-type", "float16",
    "--output-dir", "results/qa_pilot_validation",
])

pilot_csv = PROJECT_DIR / "results/qa_pilot_validation/qa_predictions_for_review.csv"
if pilot_csv.exists():
    pilot_df = pd.read_csv(pilot_csv, keep_default_na=False)
    display(pilot_df[["sample_id", "question_reference", "transcript_prediction", "model_response", "completion_tokens", "response_token_limit_hit"]])
else:
    print("Pilot CSV not found; inspect the error above before continuing.")


In [ ]:
# PHASE 7 — Reproduce the recorded 100-example QA run (CPU; several hours).
run_project([
    "scripts/evaluate_speech_qa.py",
    "--n-samples", "100",
    "--seed", "42",
    "--split", "validation",
    "--asr-backend", "indic-transcribe",
    "--llm-model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--max-new-tokens", "192",
    "--device", "cpu",
    "--compute-type", "float32",
    "--output-dir", "results/qa_indic_transcribe_cpu_100_20261024",
])


In [ ]:
# OPTIONAL PHASE 7 — Compare the same QA examples with Whisper as the ASR baseline.
run_project([
    "scripts/evaluate_speech_qa.py",
    "--n-samples", "100",
    "--seed", "42",
    "--split", "validation",
    "--asr-backend", "whisper",
    "--asr-model-size", "small",
    "--llm-model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--max-new-tokens", "192",
    "--device", "cuda",
    "--compute-type", "float16",
    "--output-dir", "results/qa_whisper_validation",
])

## Response-quality review: human labels vs AI-assisted estimates

For independently human-reviewed results, open the generated `qa_predictions_for_review.csv` in the Files panel and fill `human_correct_0_or_1`, `human_hallucination_0_or_1`, and `response_truncated_0_or_1` according to `README.md`. Use `1`/`0` and leave genuinely unreviewed rows blank. The `response_token_limit_hit` flag is a diagnostic only; visible truncation must be judged from the actual response. Ideally, a second reviewer independently labels at least 20% of rows.

The release also includes separate aggregate AI-assisted estimates (12/100 correct/adequate, 24/100 hallucination flags, 44/100 visible-truncation flags). These are explicitly provisional and must not be copied into the official `human_*` columns or called human-verified. A spot-check of 20 examples found at least one clear content-level question/reference-answer mismatch (`sample-0088`) and multiple prompts containing several tasks where the reference may cover only one component (including `sample-0010` and `sample-0020`). This does not prove a general CSV row-order shift; it means some underlying items may be unsuitable for response-quality scoring without review. Treat the aggregate estimates as exploratory until these data-quality concerns are resolved.


In [ ]:
# CELL — Score the manual review after filling the CSV columns.
run_project([
    "scripts/score_manual_review.py",
    "results/qa_indic_transcribe_cpu_100_20261024/qa_predictions_for_review.csv",
    "--output-json",
    "results/qa_indic_transcribe_cpu_100_20261024/manual_response_metrics.json",
])


In [ ]:
# FINAL CELL — Inspect bundled aggregate snapshots and outputs from this run.
from pathlib import Path
import json
import pandas as pd

bundled = [
    "results/asr_validation_summary.csv",
    "results/adaptive_threshold_sweep.csv",
    "results/asr_test_summary.csv",
    "results/qa_run_summary.json",
    "results/qa_ai_assisted_review_summary.json",
    "results/reproducibility_check.json",
]
fresh_run_outputs = [
    "results/whisper_validation_deterministic/fleurs_validation_summary.csv",
    "results/whisper_validation_threshold_analysis/validation_threshold_macro_summary.csv",
    "results/whisper_validation_threshold_analysis/validation_threshold_summary.csv",
    "results/final_test_threshold_m03/fleurs_test_summary.csv",
    "results/qa_indic_transcribe_cpu_100_20261024/qa_run_summary.json",
    "results/qa_indic_transcribe_cpu_100_20261024/manual_response_metrics.json",
]

def inspect_artifacts(title, paths):
    print("\n===", title, "===")
    for rel in paths:
        path = PROJECT_DIR / rel
        print("\n---", rel, "| exists:", path.exists(), "---")
        if not path.exists():
            continue
        if path.suffix.lower() == ".csv":
            display(pd.read_csv(path))
        elif path.suffix.lower() == ".json":
            print(path.read_text(encoding="utf-8")[:6000])

inspect_artifacts("Bundled aggregate reference snapshots", bundled)
inspect_artifacts("Outputs generated by a fresh run", fresh_run_outputs)


## Final submission checklist

- State Hindi/Kannada WER/CER by clean/noisy condition and method; distinguish per-group corpus rates from macro averages.
- Include the validation-only threshold sweep, frozen threshold -0.3, and held-out test results.
- Include end-to-end QA WER/CER and latency/RTF; note that the final 100-example QA run was CPU-only and MMS Hindi TTS was used.
- Present response-quality estimates as AI-assisted until an independent human fills the official `human_*` columns and the scorer is run.
- Record model/dataset revisions, package versions, hardware, split, seed, sample count, and temperature setting.
- Keep gated-dataset row-level QA predictions and all authentication tokens out of any public GitHub repository unless redistribution is explicitly permitted.
- No test metric was used to select the threshold. Do not claim a winning adaptive method: fixed beam-5 had the best macro test WER/CER in this benchmark.
